# 임베딩 튜닝 비교 — prefix 실험 + Hybrid (KoE5 + KURE)

**목적**: PR #21 의 KoE5 교체 후 추가 개선 시도.

1. **Prefix 실험** — KoE5 의 입력 prefix 가 단어 사전에 적절한가?
   - `query: ` (현행 — E5 표준이지만 단어 사전엔 의미적 부적절 가능)
   - `passage: ` (사전 entry 는 query 보다 passage 결)
   - 없음 (BGE 패턴)
2. **Hybrid (KoE5 + KURE)** — 두 모델 장점 결합
   - KoE5: 단일 의미 cluster (게임 적합)
   - KURE: 동음이의어 잡음 + 글자 패턴 X
   - concat (2048 dim) vs average (1024 dim)

**전제**: 아래 임베딩들 생성 후 실행:
```
# 1. KoE5 prefix 실험 (3가지)
python tools/embedding_eval/build_alt_embeddings.py \
  --model nlpai-lab/KoE5 --output data/embedding_dictionary_koe5_noprefix.json --prefix ""
python tools/embedding_eval/build_alt_embeddings.py \
  --model nlpai-lab/KoE5 --output data/embedding_dictionary_koe5_passage.json --prefix "passage: "
# (koe5.json — 'query: ' prefix — 이미 있음)

# 2. Hybrid (KoE5 concat KURE, 2048 dim)
python tools/embedding_eval/build_hybrid_embedding.py \
  --source-a data/embedding_dictionary_koe5.json \
  --source-b data/embedding_dictionary_kure.json \
  --output data/embedding_dictionary_hybrid_koe5_kure_concat.json --mode concat
```

## 1. Setup — 모든 임베딩 사전 로드

In [1]:
import json
from pathlib import Path
import numpy as np

REPO_ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents)
                 if (p / 'data' / 'embedding_dictionary_e5.json').exists())
DATA = REPO_ROOT / 'data'

def load_emb(path):
    if not path.exists():
        return None
    with open(path, 'rb') as f:
        d = json.loads(f.read())
    words = list(d.keys())
    mat = np.array([d[w] for w in words], dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    mat = mat / norms
    return words, mat, {w: i for i, w in enumerate(words)}

# 모델별로 (라벨, path) — 존재하는 것만 로드
specs = [
    ('KoE5 (query:)',     'embedding_dictionary_koe5.json'),
    ('KoE5 (passage:)',   'embedding_dictionary_koe5_passage.json'),
    ('KoE5 (no prefix)',  'embedding_dictionary_koe5_noprefix.json'),
    ('KURE',              'embedding_dictionary_kure.json'),
    ('Hybrid concat',     'embedding_dictionary_hybrid_koe5_kure_concat.json'),
    ('Hybrid avg',        'embedding_dictionary_hybrid_koe5_kure_avg.json'),
]

models = []
for label, fname in specs:
    loaded = load_emb(DATA / fname)
    if loaded is None:
        print(f'  ✗ {label}: {fname} 없음 (skip)')
        continue
    words, mat, idx = loaded
    print(f'  ✓ {label}: {len(words):,} words, dim={mat.shape[1]}')
    models.append((label, words, mat, idx))

print(f'\n총 {len(models)} 모델 로드')

  ✓ KoE5 (query:): 60,000 words, dim=1024
  ✓ KoE5 (passage:): 60,000 words, dim=1024
  ✓ KoE5 (no prefix): 60,000 words, dim=1024
  ✓ KURE: 60,000 words, dim=1024
  ✓ Hybrid concat: 60,000 words, dim=2048
  ✗ Hybrid avg: embedding_dictionary_hybrid_koe5_kure_avg.json 없음 (skip)

총 5 모델 로드


## 2. Helper

In [2]:
def pair_cos(mat, idx, a, b):
    if a not in idx or b not in idx:
        return float('nan')
    return float(mat[idx[a]] @ mat[idx[b]])

def rank_of(words, mat, idx, answer, candidates):
    if answer not in idx:
        return None
    vec = mat[idx[answer]]
    sims = mat @ vec
    order = np.argsort(-sims)
    rank_map = {words[i]: r for r, i in enumerate(order, 1)}
    return [(c, rank_map.get(c, -1), float(sims[idx[c]]) if c in idx else float('nan'))
            for c in candidates]

def top_n(words, mat, idx, answer, n=15, exclude_self=True):
    if answer not in idx:
        return None
    vec = mat[idx[answer]]
    sims = mat @ vec
    order = np.argsort(-sims)
    skip = 1 if exclude_self else 0
    return [(words[i], float(sims[i])) for i in order[skip:skip + n]]

## 3. 결정적 pair — 모델별

**합격 기준**: 사과↔배 > 사과↔자동차 + 강아지↔고양이 > 강아지↔자동차

In [3]:
pairs = [('사과', '배'), ('사과', '자동차'), ('사과', '포도'), ('사과', '딸기'),
         ('강아지', '고양이'), ('강아지', '자동차')]

header = f'{"pair":<18}'
for label, *_ in models:
    header += f' {label:>16}'
print(header)
print('-' * len(header))

for a, b in pairs:
    line = f'{a + "↔" + b:<18}'
    for label, words, mat, idx in models:
        c = pair_cos(mat, idx, a, b)
        line += f' {c:>16.4f}' if c == c else f' {"-":>16}'
    print(line)

print('\n핵심 점검:')
for label, words, mat, idx in models:
    sb = pair_cos(mat, idx, '사과', '배')
    sc = pair_cos(mat, idx, '사과', '자동차')
    ok = '✓' if sb > sc else '✗'
    print(f'  {label:<20} 사과↔배 ({sb:.4f}) {">" if sb > sc else "<"} 사과↔자동차 ({sc:.4f})  {ok}')

pair                  KoE5 (query:)  KoE5 (passage:) KoE5 (no prefix)             KURE    Hybrid concat
-------------------------------------------------------------------------------------------------------
사과↔배                         0.6450           0.6452           0.5794           0.4670           0.5560
사과↔자동차                       0.6609           0.6598           0.5635           0.4656           0.5632
사과↔포도                        0.6678           0.7175           0.5728           0.4886           0.5782
사과↔딸기                        0.6778           0.7217           0.5728           0.4483           0.5630
강아지↔고양이                      0.7816           0.8332           0.6799           0.7902           0.7859
강아지↔자동차                      0.6531           0.6699           0.5054           0.5711           0.6121

핵심 점검:
  KoE5 (query:)        사과↔배 (0.6450) < 사과↔자동차 (0.6609)  ✗
  KoE5 (passage:)      사과↔배 (0.6452) < 사과↔자동차 (0.6598)  ✗
  KoE5 (no prefix)     사과↔배 (0.5794) > 사과↔자동

## 4. "사과" top 15 — 모델별

**관찰**: 글자 패턴 cluster (-과 끝) 가 얼마나 남아있나? 의미 단어 (과일·과자) 등장 수?

In [4]:
for label, words, mat, idx in models:
    print(f'\n--- "사과" top 15 — {label} ---')
    res = top_n(words, mat, idx, '사과', 15)
    if res is None:
        print('  (사전에 없음)')
        continue
    for r, (w, s) in enumerate(res, 1):
        print(f'  {r:>2}. {w:<12} {s:.4f}')


--- "사과" top 15 — KoE5 (query:) ---
   1. 사과는          0.9613
   2. 사과를          0.9447
   3. 사과와          0.9373
   4. 사과가          0.8551
   5. 대과           0.8400
   6. 사이다          0.8185
   7. 사과나무         0.8141
   8. 무과           0.7993
   9. 제과           0.7931
  10. 사과문을         0.7824
  11. 사과의          0.7811
  12. 과일           0.7783
  13. 과일과          0.7752
  14. 설과           0.7747
  15. 과이다          0.7724

--- "사과" top 15 — KoE5 (passage:) ---
   1. 사과를          0.9447
   2. 사과와          0.9426
   3. 사과가          0.9303
   4. 사과는          0.9302
   5. 사과나무         0.8600
   6. 사이다          0.8563
   7. 대과           0.8354
   8. 오과다          0.8250
   9. 사과술          0.8214
  10. 제과           0.8184
  11. 사과문을         0.8132
  12. 과일           0.8085
  13. 무과           0.8047
  14. 꽹과리          0.8033
  15. 과실           0.8017

--- "사과" top 15 — KoE5 (no prefix) ---
   1. 사과와          0.8507
   2. 사과는          0.8164
   3. 대과           0.8147
   4. 사과를          0.8089


## 5. 과일 cross-check — 모델별

**합격 기준**: 과일들이 top 100 안에 많이 들어와야 정상

In [5]:
FRUITS = ['배', '포도', '바나나', '딸기', '수박', '망고', '레몬', '복숭아']
ANSWER = '사과'

header = f'{"fruit":<8}'
for label, *_ in models:
    header += f' {label:>16}'
print(f'정답: "{ANSWER}" — 과일들의 rank\n')
print(header)
print('-' * len(header))

all_ranks = {}
for label, words, mat, idx in models:
    all_ranks[label] = rank_of(words, mat, idx, ANSWER, FRUITS)

for i, fruit in enumerate(FRUITS):
    line = f'{fruit:<8}'
    for label, *_ in models:
        ranks = all_ranks[label]
        if ranks is None:
            line += f' {"-":>16}'
            continue
        _, r, _ = ranks[i]
        line += f' {r:>16,}' if r > 0 else f' {"없음":>16}'
    print(line)

# top 100 안에 들어온 개수
print('\ntop 100 안 과일 개수:')
for label, *_ in models:
    ranks = all_ranks[label]
    if ranks is None:
        print(f'  {label}: -')
        continue
    count = sum(1 for _, r, _ in ranks if 0 < r <= 100)
    print(f'  {label:<20} {count}/{len(FRUITS)}')

# 평균 rank
print('\n평균 rank (작을수록 좋음):')
for label, *_ in models:
    ranks = all_ranks[label]
    if ranks is None:
        print(f'  {label}: -')
        continue
    vals = [r for _, r, _ in ranks if r > 0]
    avg = sum(vals) / len(vals) if vals else 0
    print(f'  {label:<20} {avg:>10,.0f}')

정답: "사과" — 과일들의 rank

fruit       KoE5 (query:)  KoE5 (passage:) KoE5 (no prefix)             KURE    Hybrid concat
---------------------------------------------------------------------------------------------
배                   2,590            8,588            1,994           17,708            4,190
포도                    874              264            2,664            9,732            1,126
바나나                   118               66            5,111           15,001              513
딸기                    563              217            2,669           26,179            2,739
수박                    244               30              117           27,015            1,666
망고                    472              131            2,829           24,150            2,138
레몬                    453               43              594           25,016            2,206
복숭아                 1,099              269            8,019           38,766            9,219

top 100 안 과일 개수:
  KoE5 (query:)     

## 6. 강아지 / 학교 회귀 확인

정상 cluster 가 깨지지 않았는지 확인.

In [6]:
for ans, cands in [
    ('강아지', ['고양이', '애완동물', '반려견', '동물']),
    ('학교',   ['대학', '교실', '교사', '교육']),
]:
    print(f'\n정답 "{ans}" — 기대 단어 rank:')
    header = f'  {"word":<8}'
    for label, *_ in models:
        header += f' {label:>16}'
    print(header)
    
    rank_data = {label: rank_of(w, m, i, ans, cands) for label, w, m, i in models}
    for j, c in enumerate(cands):
        line = f'  {c:<8}'
        for label, *_ in models:
            r = rank_data[label]
            if r is None:
                line += f' {"-":>16}'
                continue
            _, rank, _ = r[j]
            line += f' {rank:>16,}' if rank > 0 else f' {"없음":>16}'
        print(line)


정답 "강아지" — 기대 단어 rank:
  word        KoE5 (query:)  KoE5 (passage:) KoE5 (no prefix)             KURE    Hybrid concat
  고양이                    14               13               50               10               10
  애완동물                    3               11                2                4                2
  반려견                     7                4               16                2                3
  동물                     18               18               42                6                7

정답 "학교" — 기대 단어 rank:
  word        KoE5 (query:)  KoE5 (passage:) KoE5 (no prefix)             KURE    Hybrid concat
  대학                     29               17                5               23               23
  교실                     28               33               51               25               25
  교사                     72               79              170               49               53
  교육                     14               27                3               32           

## 7. 종합 결정 (사용자 채움)

### 7.1 prefix 영향
- KoE5 query vs passage vs 없음 — 어떤 게 가장 좋나? : ___
- 차이 큰가 작은가? : ___

### 7.2 hybrid 효과
- KoE5 단독 vs Hybrid concat — 과일 rank 개선되나? : ___
- 회귀 (강아지·학교) 있나? : ___
- 결정적 pair (사과↔배 > 사과↔자동차) 합격? : ___

### 7.3 최종 선정
- [ ] KoE5 (query:) 그대로 — 다른 옵션 다 비슷하거나 못함
- [ ] KoE5 (passage:) — prefix 변경만으로 개선
- [ ] KoE5 (no prefix) — prefix 자체가 noise
- [ ] Hybrid concat — 두 모델 결합이 best
- [ ] Hybrid avg — concat 너무 무거우면 (2048 dim) average 절충

### 7.4 다음 액션
선정한 임베딩으로 `data/embedding_dictionary_e5.json` swap 후 게임 테스트.